In [2]:
try:
    import sksurv

    print("scikit-survival is installed.")
    print("Version:", sksurv.__version__)

except ImportError:
    print("scikit-survival is not installed.")

scikit-survival is installed.
Version: 0.28.0


In [3]:
import pandas as pd
import numpy as np

survival_path = "../data/processed/survival_dataset.csv"

survival_df = pd.read_csv(survival_path)

print("Dataset shape:", survival_df.shape)
display(survival_df.head())
print(survival_df.columns.tolist())

Dataset shape: (50000, 5)


,customer_id,duration_days,duration_years,event,annual_premium
0,2.213007e+11,16,0.043806,1,756.722144
1,2.213033e+11,4828,13.218344,0,875.814547
2,2.213023e+11,1751,4.793977,0,654.552197
3,2.213030e+11,2230,6.105407,0,795.654884
4,2.213006e+11,159,0.435318,1,1142.303520


['customer_id', 'duration_days', 'duration_years', 'event', 'annual_premium']


In [4]:
from sklearn.model_selection import train_test_split

# Features used by the initial survival model
feature_columns = [
    "annual_premium"
]

X = survival_df[feature_columns].copy()

# Convert event and duration into numeric values
y_duration = survival_df["duration_years"].astype(float)
y_event = survival_df["event"].astype(int)

print("Feature shape:", X.shape)
print("Duration shape:", y_duration.shape)
print("Event shape:", y_event.shape)

print("\nEvent distribution:")
print(y_event.value_counts())

Feature shape: (50000, 1)
Duration shape: (50000,)
Event shape: (50000,)

Event distribution:
event
0    45816
1     4184
Name: count, dtype: int64


In [5]:
from sksurv.util import Surv

y_survival = Surv.from_arrays(
    event=y_event.astype(bool),
    time=y_duration
)

print("Survival target created successfully.")
print("Target shape:", y_survival.shape)
print("First five targets:")
print(y_survival[:5])

Survival target created successfully.
Target shape: (50000,)
First five targets:
[( True,  0.04380561) (False, 13.2183436 ) (False,  4.79397673)
 (False,  6.10540726) ( True,  0.43531828)]


In [6]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y_survival,
    test_size=0.30,
    random_state=42,
    stratify=y_event
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("y_train:", y_train.shape)
print("y_test:", y_test.shape)

X_train: (35000, 1)
X_test: (15000, 1)
y_train: (35000,)
y_test: (15000,)


In [7]:
from sksurv.ensemble import RandomSurvivalForest

print("Random Survival Forest imported successfully.")

Random Survival Forest imported successfully.


In [8]:
# Create Random Survival Forest model
rsf_model = RandomSurvivalForest(
    n_estimators=200,
    min_samples_split=10,
    min_samples_leaf=15,
    max_features="sqrt",
    n_jobs=-1,
    random_state=42
)

# Train the model
rsf_model.fit(X_train, y_train)

print("Random Survival Forest model trained successfully.")

MemoryError: could not allocate 161480704 bytes

In [ ]:
# Evaluate the model on training data
train_c_index = rsf_model.score(X_train, y_train)

# Evaluate the model on testing data
test_c_index = rsf_model.score(X_test, y_test)

print("Random Survival Forest Results")
print("--------------------------------")
print("Training Concordance Index:", round(train_c_index, 4))
print("Testing Concordance Index:", round(test_c_index, 4))

In [ ]:
# Generate risk scores
train_risk_scores = rsf_model.predict(X_train)
test_risk_scores = rsf_model.predict(X_test)

print("Training risk score shape:", train_risk_scores.shape)
print("Testing risk score shape:", test_risk_scores.shape)

print("\nFirst five testing risk scores:")
print(test_risk_scores[:5])

In [ ]:
import os
import joblib

# Create model directory if it does not exist
os.makedirs("../models_saved", exist_ok=True)

# Save the trained RSF model
rsf_model_path = "../models_saved/random_survival_forest.pkl"

joblib.dump(rsf_model, rsf_model_path)

print("Random Survival Forest model saved successfully.")
print("Saved path:", rsf_model_path)

In [ ]:
print("Model type:", type(rsf_model).__name__)
print("Number of trees:", rsf_model.n_estimators)
print("Model saved at:", rsf_model_path)